# 05 — Team Strength Model

This notebook combines recent team performance with projected 2026 personnel strength to create a single preseason rating for each NFL team.

Historical team performance provides the baseline estimate of team quality, while quarterback, skill position, offensive line, and defensive projections adjust that baseline for the roster entering 2026.

The final team strength ratings will be used in Notebook 06 to generate game-level win probabilities.

In [1]:
from pathlib import Path

import pandas as pd
import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

In [2]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

In [3]:
historical_team_features = pl.read_parquet(
    PROCESSED_DIR / "historical_team_features.parquet"
)

qb_2026 = pl.read_parquet(
    PROCESSED_DIR / "2026_qb_strength.parquet"
)

skill_2026 = pl.read_parquet(
    PROCESSED_DIR / "2026_skill_position_strength.parquet"
)

ol_2026 = pl.read_parquet(
    PROCESSED_DIR / "2026_ol_strength.parquet"
)

defense_2026 = pl.read_parquet(
    PROCESSED_DIR / "2026_defensive_strength.parquet"
)

roster_continuity_2026 = pl.read_parquet(
    PROCESSED_DIR / "2026_roster_continuity.parquet"
)

In [4]:
roster_continuity_2026_pd = (
    roster_continuity_2026.to_pandas()
)

## Historical Team Baseline

A team's preseason strength should not depend entirely on its most recent season, since single season NFL performance can be volatile.

The historical baseline therefore uses point differential per game from the previous three seasons, weighted toward the most recent year:

- 60% previous season
- 25% two seasons ago
- 15% three seasons ago

A simple linear regression then learns how that weighted historical performance translates into the following season's point differential. This approach remains leakage-safe while reducing overreaction to one unusually strong or weak season.

In [5]:
team_history = (
    historical_team_features
    .select([
        "season",
        "team",
        "point_diff_per_game"
    ])
    .to_pandas()
    .sort_values([
        "team",
        "season"
    ])
)

team_history["pd_lag1"] = (
    team_history
    .groupby("team")["point_diff_per_game"]
    .shift(1)
)

team_history["pd_lag2"] = (
    team_history
    .groupby("team")["point_diff_per_game"]
    .shift(2)
)

team_history["pd_lag3"] = (
    team_history
    .groupby("team")["point_diff_per_game"]
    .shift(3)
)

team_history["weighted_prior_pd"] = (
    0.60 * team_history["pd_lag1"]
    + 0.25 * team_history["pd_lag2"]
    + 0.15 * team_history["pd_lag3"]
)

In [6]:
baseline_training = (
    team_history
    .dropna(
        subset=[
            "weighted_prior_pd",
            "point_diff_per_game"
        ]
    )
    .copy()
)

baseline_model = LinearRegression()

baseline_model.fit(
    baseline_training[
        ["weighted_prior_pd"]
    ],
    baseline_training[
        "point_diff_per_game"
    ]
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[0.54]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['weighted_prior_pd']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.001751
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(1)


### 2026 Historical Baseline

For the 2026 projection, the historical input combines each team's 2025, 2024, and 2023 point differential using the same 60/25/15 weighting. The fitted historical model converts that recent performance into an expected 2026 baseline strength.

In [7]:
pd_2025 = (
    historical_team_features
    .filter(pl.col("season") == 2025)
    .select([
        "team",
        "point_diff_per_game"
    ])
    .rename({
        "point_diff_per_game": "pd_2025"
    })
    .to_pandas()
)

pd_2024 = (
    historical_team_features
    .filter(pl.col("season") == 2024)
    .select([
        "team",
        "point_diff_per_game"
    ])
    .rename({
        "point_diff_per_game": "pd_2024"
    })
    .to_pandas()
)

pd_2023 = (
    historical_team_features
    .filter(pl.col("season") == 2023)
    .select([
        "team",
        "point_diff_per_game"
    ])
    .rename({
        "point_diff_per_game": "pd_2023"
    })
    .to_pandas()
)

team_2026_baseline = (
    pd_2025
    .merge(
        pd_2024,
        on="team",
        how="inner"
    )
    .merge(
        pd_2023,
        on="team",
        how="inner"
    )
)

team_2026_baseline["weighted_prior_pd"] = (
    0.60 * team_2026_baseline["pd_2025"]
    + 0.25 * team_2026_baseline["pd_2024"]
    + 0.15 * team_2026_baseline["pd_2023"]
)

team_2026_baseline["baseline_team_strength"] = (
    baseline_model.predict(
        team_2026_baseline[
            ["weighted_prior_pd"]
        ]
    )
)

team_2026_baseline = team_2026_baseline[
    [
        "team",
        "baseline_team_strength"
    ]
].copy()

In [8]:
baseline_training["predicted_point_diff_per_game"] = (
    baseline_model.predict(
        baseline_training[["weighted_prior_pd"]]
    )
)

mae = mean_absolute_error(
    baseline_training["point_diff_per_game"],
    baseline_training["predicted_point_diff_per_game"]
)

correlation = baseline_training[
    [
        "point_diff_per_game",
        "predicted_point_diff_per_game"
    ]
].corr().iloc[0, 1]

print(f"Training rows: {len(baseline_training)}")
print(f"MAE: {mae:.3f} points per game")
print(f"Correlation: {correlation:.3f}")

print()
print("Model coefficients:")
print(
    f"weighted_prior_pd: "
    f"{baseline_model.coef_[0]:.3f}"
)
print(
    f"Intercept: "
    f"{baseline_model.intercept_:.3f}"
)

Training rows: 256
MAE: 4.484 points per game
Correlation: 0.433

Model coefficients:
weighted_prior_pd: 0.540
Intercept: 0.002


### Season-Based Validation

Because NFL performance changes over time, the baseline is evaluated by training on earlier seasons and predicting later seasons.

This provides a more realistic test than evaluating the model on the same observations used to fit it. Each validation season is predicted using only team-seasons that occurred before it.

In [9]:
validation_results = []

validation_seasons = sorted(
    season
    for season in baseline_training["season"].unique()
    if 2020 <= season <= 2025
)

for season in validation_seasons:
    train = baseline_training[
        baseline_training["season"] < season
    ]

    test = baseline_training[
        baseline_training["season"] == season
    ]

    if train.empty or test.empty:
        continue

    model = LinearRegression()

    model.fit(
        train[["weighted_prior_pd"]],
        train["point_diff_per_game"]
    )

    predictions = model.predict(
        test[["weighted_prior_pd"]]
    )

    validation_results.append({
        "season": season,
        "mae": mean_absolute_error(
            test["point_diff_per_game"],
            predictions
        ),
        "correlation": test[
            "point_diff_per_game"
        ].corr(
            pd.Series(
                predictions,
                index=test.index
            )
        )
    })

validation_results = pd.DataFrame(
    validation_results
)

print(validation_results.round(3))

print()

if not validation_results.empty:
    print(
        "Average MAE:",
        round(
            validation_results["mae"].mean(),
            3
        )
    )

    print(
        "Average correlation:",
        round(
            validation_results["correlation"].mean(),
            3
        )
    )

   season    mae  correlation
0    2020  4.965        0.401
1    2021  4.492        0.585
2    2022  4.250        0.276
3    2023  3.846        0.564
4    2024  5.242        0.291
5    2025  4.445        0.356

Average MAE: 4.54
Average correlation: 0.412


## 2026 Personnel Strength

The baseline reflects how each team performed in 2025, but NFL rosters change substantially between seasons.

The 2026 personnel projections provide information about the roster entering the new season. Quarterback is kept separate because of its outsized influence, while RB, WR, TE, offensive line, front seven, and secondary ratings represent the major supporting position groups.

In [10]:
qb_2026_pd = qb_2026.to_pandas()
skill_2026_pd = skill_2026.to_pandas()
ol_2026_pd = ol_2026.to_pandas()
defense_2026_pd = defense_2026.to_pandas()

In [11]:
team_2026 = (
    team_2026_baseline
    .merge(
        qb_2026_pd[
            [
                "team",
                "qb1_projected_epa_per_dropback"
            ]
        ],
        on="team",
        how="left"
    )
    .merge(
        skill_2026_pd,
        on="team",
        how="left"
    )
    .merge(
        ol_2026_pd,
        on="team",
        how="left"
    )
    .merge(
        defense_2026_pd,
        on="team",
        how="left"
    )
    .merge(
        roster_continuity_2026_pd,
        on="team",
        how="left"
    )
)

In [12]:
print("Rows:", len(team_2026))
print("Unique teams:", team_2026["team"].nunique())
print()
print("Missing values:")
print(team_2026.isna().sum())

print()
print(
    team_2026
    .sort_values(
        "baseline_team_strength",
        ascending=False
    )
    .head(10)
)

Rows: 32
Unique teams: 32

Missing values:
team                              0
baseline_team_strength            0
qb1_projected_epa_per_dropback    0
rb_strength_z                     0
wr_strength_z                     0
te_strength_z                     0
ol_protection_strength            0
front_seven_strength_z            0
secondary_strength_z              0
roster_continuity                 0
dtype: int64

   team  baseline_team_strength  qb1_projected_epa_per_dropback  \
3   BUF                4.125764                        0.120673   
27  SEA                3.515976                        0.087481   
10  DET                3.374644                        0.140400   
16   LA                3.257133                        0.130037   
2   BAL                2.710864                        0.102775   
9   DEN                2.355154                        0.067427   
25  PHI                2.324982                        0.063395   
12  HOU                2.193179                

## Personnel Adjustment

The 2026 personnel ratings are standardized so that each position group is measured on a comparable scale.

Quarterback receives the largest weight because of its influence on team performance. The remaining weight is distributed across skill positions, offensive line, front seven, and secondary. The resulting personnel score is used as an adjustment to the historical team baseline rather than as a standalone prediction.

In [13]:
team_2026["qb_strength_z"] = (
    team_2026["qb1_projected_epa_per_dropback"]
    - team_2026["qb1_projected_epa_per_dropback"].mean()
) / team_2026["qb1_projected_epa_per_dropback"].std()

team_2026["skill_strength_z"] = (
    0.25 * team_2026["rb_strength_z"]
    + 0.50 * team_2026["wr_strength_z"]
    + 0.25 * team_2026["te_strength_z"]
)

In [14]:
team_2026["personnel_strength"] = (
    0.40 * team_2026["qb_strength_z"]
    + 0.15 * team_2026["skill_strength_z"]
    + 0.15 * team_2026["ol_protection_strength"]
    + 0.15 * team_2026["front_seven_strength_z"]
    + 0.15 * team_2026["secondary_strength_z"]
)

### Roster Continuity Adjustment

Personnel quality measures how strong the current roster appears to be, while roster continuity measures how much of the previous team remains intact.

Continuity is treated as a smaller adjustment because returning players can improve stability and reduce uncertainty, but continuity alone does not guarantee a strong roster.

In [15]:
team_2026["roster_continuity_z"] = (
    team_2026["roster_continuity"]
    - team_2026["roster_continuity"].mean()
) / team_2026["roster_continuity"].std()

In [16]:
ROSTER_CONTINUITY_POINT_VALUE = 0.5

team_2026["roster_continuity_adjustment"] = (
    ROSTER_CONTINUITY_POINT_VALUE
    * team_2026["roster_continuity_z"]
)

## Final 2026 Team Strength

The personnel score is converted into a point based adjustment and added to the historical baseline.

A one standard deviation difference in overall personnel strength is worth approximately 1.5 points per game. This allows meaningful roster differences to move teams while keeping recent team performance as the primary foundation of the projection.

In [17]:
PERSONNEL_POINT_VALUE = 1.5
PERSONNEL_WEIGHT = 0.60
CONTINUITY_WEIGHT = 0.50

team_2026["personnel_adjustment"] = (
    PERSONNEL_POINT_VALUE
    * team_2026["personnel_strength"]
)

team_2026["team_strength"] = (
    team_2026["baseline_team_strength"]
    + PERSONNEL_WEIGHT
    * team_2026["personnel_adjustment"]
    + CONTINUITY_WEIGHT
    * team_2026["roster_continuity_adjustment"]
)

In [18]:
team_strength_review = (
    team_2026[
        [
            "team",
            "baseline_team_strength",
            "personnel_strength",
            "personnel_adjustment",
            "roster_continuity",
            "roster_continuity_adjustment",
            "team_strength"
        ]
    ]
    .sort_values(
        "team_strength",
        ascending=False
    )
)

print(team_strength_review.round(3).to_string(index=False))

print()
print(
    "Personnel adjustment range:",
    round(team_2026["personnel_adjustment"].min(), 3),
    "to",
    round(team_2026["personnel_adjustment"].max(), 3)
)

team  baseline_team_strength  personnel_strength  personnel_adjustment  roster_continuity  roster_continuity_adjustment  team_strength
 BUF                   4.126               0.623                 0.935              0.573                         0.260          4.817
  LA                   3.257               1.401                 2.101              0.609                         0.490          4.762
 SEA                   3.516               0.424                 0.636              0.677                         0.932          4.363
 DET                   3.375               0.695                 1.042              0.538                         0.038          4.019
 DEN                   2.355               0.653                 0.980              0.727                         1.253          3.570
 BAL                   2.711               0.424                 0.636              0.527                        -0.037          3.074
 HOU                   2.193               0.783       

## Final Team Strength Ratings

The final rating represents expected team quality on a neutral field entering the 2026 season.

It combines the historical team-performance baseline with the 2026 personnel adjustment. Schedule, home field advantage, and game specific context are intentionally excluded here and will be incorporated when individual games are modeled in Notebook 06.

In [19]:
team_strength_2026 = (
    team_2026[
        [
            "team",
            "baseline_team_strength",
            "personnel_strength",
            "personnel_adjustment",
            "roster_continuity",
            "roster_continuity_adjustment",
            "team_strength"
        ]
    ]
    .sort_values(
        "team_strength",
        ascending=False
    )
    .reset_index(drop=True)
)

team_strength_2026["strength_rank"] = (
    team_strength_2026.index + 1
)

team_strength_2026 = team_strength_2026[
    [
        "strength_rank",
        "team",
        "team_strength",
        "baseline_team_strength",
        "personnel_adjustment",
        "personnel_strength",
        "roster_continuity",
        "roster_continuity_adjustment"
    ]
]

In [20]:
team_strength_2026.to_parquet(
    PROCESSED_DIR / "2026_team_strength.parquet",
    index=False
)

In [21]:
print(
    f"Final team ratings: {len(team_strength_2026)} teams, "
    f"{team_strength_2026['team'].nunique()} unique teams"
)

team_strength_2026

Final team ratings: 32 teams, 32 unique teams


,strength_rank,team,team_strength,baseline_team_strength,personnel_adjustment,personnel_strength,roster_continuity,roster_continuity_adjustment
0,1,BUF,4.816514,4.125764,0.934506,0.623004,0.573034,0.260091
1,2,LA,4.762452,3.257133,2.100811,1.400540,0.608696,0.489665
2,3,SEA,4.363421,3.515976,0.635683,0.423788,0.677419,0.932073
3,4,DET,4.018565,3.374644,1.041923,0.694616,0.538462,0.037533
4,5,DEN,3.569507,2.355154,0.979753,0.653169,0.727273,1.253003
5,6,BAL,3.074111,2.710864,0.636254,0.424170,0.526882,-0.037012
6,7,HOU,2.810133,2.193179,1.173833,0.782555,0.505495,-0.174691
7,8,GB,2.536317,1.718369,0.882628,0.588419,0.622222,0.576742
8,9,PHI,2.451919,2.324982,0.264690,0.176460,0.522727,-0.063756
9,10,SF,2.366788,1.805709,0.692939,0.461959,0.577778,0.290631


In [22]:
team_strength_review.loc[
    team_strength_review["team"] == "IND",
    [
        "team",
        "baseline_team_strength",
        "personnel_strength",
        "personnel_adjustment",
        "roster_continuity",
        "roster_continuity_adjustment",
        "team_strength",
    ]
]

,team,baseline_team_strength,personnel_strength,personnel_adjustment,roster_continuity,roster_continuity_adjustment,team_strength
13,IND,0.543256,-0.111709,-0.167564,0.56044,0.179017,0.532226


In [23]:
team_strength_review[
    [
        "team",
        "baseline_team_strength",
        "personnel_strength",
        "personnel_adjustment",
        "roster_continuity_adjustment",
        "team_strength",
    ]
].sort_values(
    "team_strength",
    ascending=False
).reset_index(drop=True)

,team,baseline_team_strength,personnel_strength,personnel_adjustment,roster_continuity_adjustment,team_strength
0,BUF,4.125764,0.623004,0.934506,0.260091,4.816514
1,LA,3.257133,1.400540,2.100811,0.489665,4.762452
2,SEA,3.515976,0.423788,0.635683,0.932073,4.363421
3,DET,3.374644,0.694616,1.041923,0.037533,4.018565
4,DEN,2.355154,0.653169,0.979753,1.253003,3.569507
5,BAL,2.710864,0.424170,0.636254,-0.037012,3.074111
6,HOU,2.193179,0.782555,1.173833,-0.174691,2.810133
7,GB,1.718369,0.588419,0.882628,0.576742,2.536317
8,PHI,2.324982,0.176460,0.264690,-0.063756,2.451919
9,SF,1.805709,0.461959,0.692939,0.290631,2.366788


In [24]:
team_strength_review[
    [
        "team",
        "baseline_team_strength",
        "personnel_strength",
        "personnel_adjustment",
        "roster_continuity_adjustment",
        "team_strength",
    ]
].sort_values("team_strength").head(10).round(3)

,team,baseline_team_strength,personnel_strength,personnel_adjustment,roster_continuity_adjustment,team_strength
30,TEN,-5.174,-1.382,-2.073,-0.660,-6.747
24,NYJ,-4.805,-1.157,-1.735,-0.175,-5.934
18,LV,-4.626,-0.883,-1.325,-0.545,-5.693
4,CAR,-3.703,-0.882,-1.323,-0.073,-4.533
7,CLE,-3.147,-0.923,-1.385,-0.344,-4.150
0,ARI,-2.961,-0.778,-1.167,-0.033,-3.678
23,NYG,-2.903,-0.458,-0.687,-0.282,-3.456
19,MIA,-1.116,-0.632,-0.948,-1.517,-2.443
31,WAS,-1.963,0.014,0.021,-0.104,-2.002
22,NO,-1.585,-0.075,-0.113,0.192,-1.556


In [25]:
print("TEAM STRENGTH DISTRIBUTION")
print(
    team_strength_review["team_strength"]
    .describe()
    .round(3)
)

print()
print("COMPONENT STANDARD DEVIATIONS")

for col in [
    "baseline_team_strength",
    "personnel_adjustment",
    "roster_continuity_adjustment",
    "team_strength",
]:
    print(
        f"{col}: "
        f"{team_strength_review[col].std():.3f}"
    )

TEAM STRENGTH DISTRIBUTION
count    32.000
mean      0.002
std       3.266
min      -6.747
25%      -2.112
50%       0.538
75%       2.473
max       4.817
Name: team_strength, dtype: float64

COMPONENT STANDARD DEVIATIONS
baseline_team_strength: 2.634
personnel_adjustment: 0.965
roster_continuity_adjustment: 0.500
team_strength: 3.266
